# Entity feature extraction (OpenMed → BioBERT)

Batch-extract disease/drug entities from chief complaints, build auxiliary features, and prepare **text-prefix enrichment** (experiment A) for SMOTE fine-tuning.

**Prerequisites:** `train.csv`, `chief_complaints.csv` in `fine-tuned-biobert/`, `pip install openmed==1.7.0`

See [`OPENMED.md`](../OPENMED.md) and [`scripts/evaluate_triage.py`](../scripts/evaluate_triage.py).

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ML_DIR = ROOT.parent / "curatio" / "server" / "ml"
sys.path.insert(0, str(ML_DIR))
os.environ["OPENMED_ENABLED"] = "true"

from openmed_enrich import analyze_entities, build_entity_prefix

TRAIN_CSV = ROOT / "train.csv"
COMPLAINTS_CSV = ROOT / "chief_complaints.csv"
DEID_CSV = ROOT / "chief_complaints_deidentified.csv"
OUTPUT_PARQUET = ROOT / "entity_features.parquet"
OUTPUT_PREFIX_CSV = ROOT / "chief_complaints_with_entity_prefix.csv"

HIGH_URGENCY_TERMS = {
    "stemi", "anaphylaxis", "hemorrhage", "hemorrhagic", "pneumothorax",
    "meningococcal", "dissection", "thunderclap", "sepsis", "arrest",
}

In [ ]:
def load_complaints() -> pd.DataFrame:
    complaints_path = DEID_CSV if DEID_CSV.exists() else COMPLAINTS_CSV
    text_col = "chief_complaint_deidentified" if complaints_path == DEID_CSV else "chief_complaint_raw"
    complaints = pd.read_csv(complaints_path)
    train = pd.read_csv(TRAIN_CSV)[["patient_id", "triage_acuity"]]
    df = train.merge(complaints, on="patient_id")
    return df.rename(columns={text_col: "text"})[["patient_id", "text", "triage_acuity"]]


def extract_row_features(text: str) -> dict:
    entities = analyze_entities(text)
    disease_terms = [e["text"].lower() for e in entities["diseases"] if not e.get("negated")]
    drug_terms = [e["text"].lower() for e in entities["drugs"] if not e.get("negated")]
    all_terms = set(disease_terms + drug_terms)
    return {
        "disease_count": entities["disease_count"],
        "drug_count": entities["drug_count"],
        "has_negated_critical_symptom": entities["has_negated_critical_symptom"],
        "high_urgency_entity_hit": int(any(t in HIGH_URGENCY_TERMS or any(u in t for u in HIGH_URGENCY_TERMS) for t in all_terms)),
        "entity_prefix": build_entity_prefix(entities),
        "diseases_json": str(entities["diseases"]),
        "drugs_json": str(entities["drugs"]),
    }


df = load_complaints()
print(f"Loaded {len(df):,} labeled complaints")

In [ ]:
# Smoke test on 3 rows before full batch (OpenMed is slow on CPU)
for row in df.head(3).itertuples():
    feats = extract_row_features(row.text)
    prefixed = f"{feats['entity_prefix']} {row.text}".strip() if feats["entity_prefix"] else row.text
    print(row.patient_id, feats["disease_count"], feats["drug_count"], prefixed[:120], "...")

In [ ]:
# Full batch — set LIMIT=None for all 80k rows (run on GPU machine or overnight)
LIMIT = 200  # increase after smoke test

subset = df.head(LIMIT).copy()
feature_rows = []
for i, row in enumerate(subset.itertuples(), start=1):
    feats = extract_row_features(row.text)
    feats["patient_id"] = row.patient_id
    feats["triage_acuity"] = row.triage_acuity
    feats["text_original"] = row.text
    prefix = feats.pop("entity_prefix")
    feats["text_with_prefix"] = f"{prefix} {row.text}".strip() if prefix else row.text
    feature_rows.append(feats)
    if i % 25 == 0:
        print(f"Processed {i}/{len(subset)}")

features = pd.DataFrame(feature_rows)
features.to_parquet(OUTPUT_PARQUET, index=False)
features[["patient_id", "text_with_prefix"]].rename(
    columns={"text_with_prefix": "chief_complaint_with_prefix"}
).to_csv(OUTPUT_PREFIX_CSV, index=False)

print(f"Wrote {OUTPUT_PARQUET} and {OUTPUT_PREFIX_CSV}")
features.describe(include="all")

## Next steps (experiment A)

1. In the SMOTE Colab notebook, pass `deidentified_csv=config.DEIDENTIFIED_COMPLAINTS_CSV` to `data_merge.merge_and_encode`, **or** merge `chief_complaints_with_entity_prefix.csv` as the text column.
2. Re-train BioBERT with prefixed text.
3. Compare checkpoints with `python scripts/evaluate_triage.py`.
4. If macro-F1 / L1 recall improve, enable `OPENMED_ENTITY_PREFIX=true` in inference.